# Q3 CVD, genotype and LD figures

**Question:** How are the existing CVD interactions, extraction completeness, LD and pooled QC displayed?

**Inputs:** Existing extraction, additive-model and genetic-QC aggregate output directories.

**Outputs:** Four source figure families and their captions/manifest.

Former CRP plots and their required input-directory checks were removed. This notebook does not fit models and does not require a CRP30 run. Use pending-extension notebook 02 only after the new CRP30 models have actually completed.


## Figure configuration


In [ ]:
from pathlib import Path
import os
import sys
import math
import json
import textwrap
from datetime import datetime

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.ticker import PercentFormatter
except Exception as exc:
    raise RuntimeError(
        "This notebook needs numpy, pandas, and matplotlib. "
        "Run it in AoU/Jupyter or another scientific Python environment. "
        f"Original import error: {exc}"
    )

# Optional overrides. Leave blank when running in the standard AoU workspace output layout.
HAIL_DIR_OVERRIDE = os.environ.get("Q3_VDR_HAIL_DIR", "")
MODEL15_DIR_OVERRIDE = os.environ.get("Q3_VDR_MODEL15_DIR", "")
STEP17_DIR_OVERRIDE = os.environ.get("Q3_VDR_STEP17_DIR", "")
OUTPUT_DIR_OVERRIDE = os.environ.get("Q3_VDR_FIGURE_OUTPUT_DIR", "")

def existing_dir(path):
    if not path:
        return None
    p = Path(path).expanduser()
    return p if p.exists() and p.is_dir() else None

def first_existing_dir(candidates, required_files=None):
    required_files = required_files or []
    for candidate in candidates:
        p = existing_dir(candidate)
        if p is None:
            continue
        if all((p / name).exists() for name in required_files):
            return p
    return None

data_dir_candidates = [
    os.environ.get("Q3_VDR_DATA_DIR", ""),
    "/home/dataproc/workspace/VDR",
    Path.cwd(),
    Path.cwd().parent,
]
DATA_DIR = first_existing_dir([p for p in data_dir_candidates if p]) or Path.cwd()

HAIL_DIR = first_existing_dir([
    HAIL_DIR_OVERRIDE,
    DATA_DIR / "Q3_VDR_tier1_hail_genotype_outputs_2026_08_07",
], ["14_hail_stage_once_local_validation.csv", "14_hail_stage_once_status.csv"])
MODEL15_DIR = first_existing_dir([
    MODEL15_DIR_OVERRIDE,
    DATA_DIR / "Q3_VDR_tier1_additive_genotype_outputs_2026_08_08",
], ["14_additive_primary_interaction_results.csv", "09_additive_vitd_dosage_cell_counts.csv"])
STEP17_DIR = first_existing_dir([
    STEP17_DIR_OVERRIDE,
    DATA_DIR / "Q3_VDR_step5_remaining_validation_outputs_2026_08_09",
], ["17_tier1_vdr_aou_ref_alt_records_for_harmonization.csv", "17_tier1_vdr_pooled_genotype_qc_hwe_descriptive.csv", "17_tier1_vdr_pairwise_ld_r2_dprime_unphased_em.csv"])

missing_dirs = [name for name, value in {
    "HAIL_DIR": HAIL_DIR,
    "MODEL15_DIR": MODEL15_DIR,
    "STEP17_DIR": STEP17_DIR,
}.items() if value is None]
if missing_dirs:
    raise FileNotFoundError(
        "Could not infer required result directories: " + ", ".join(missing_dirs) +
        ". Set Q3_VDR_HAIL_DIR, Q3_VDR_MODEL15_DIR, and/or Q3_VDR_STEP17_DIR."
    )

if OUTPUT_DIR_OVERRIDE:
    OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE).expanduser()
else:
    OUTPUT_DIR = DATA_DIR / "Q3_VDR_manuscript_figures_2026_08_09"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SNP_ORDER = ["rs2228570", "rs1544410", "rs7975232", "rs731236"]
SNP_LABELS = {
    "rs2228570": "FokI\nrs2228570",
    "rs1544410": "BsmI\nrs1544410",
    "rs7975232": "ApaI\nrs7975232",
    "rs731236": "TaqI\nrs731236",
}
SNP_SHORT = {"rs2228570": "FokI", "rs1544410": "BsmI", "rs7975232": "ApaI", "rs731236": "TaqI"}

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

FIGURE_MANIFEST = []

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")
    return path

def read_csv(path):
    return pd.read_csv(require_file(path))

def save_figure(fig, stem, caption, figure_type="main_or_supplement"):
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    FIGURE_MANIFEST.append({
        "figure_stem": stem,
        "png": str(png),
        "pdf": str(pdf),
        "figure_type": figure_type,
        "caption": caption,
    })
    print(f"Saved {png}")
    print(f"Saved {pdf}")

print("DATA_DIR:", DATA_DIR)
print("HAIL_DIR:", HAIL_DIR)
print("MODEL15_DIR:", MODEL15_DIR)
print("STEP17_DIR:", STEP17_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


## Read aggregate inputs


In [ ]:
hail_validation = read_csv(HAIL_DIR / "14_hail_stage_once_local_validation.csv")
hail_status = read_csv(HAIL_DIR / "14_hail_stage_once_status.csv")
cvd_primary = read_csv(MODEL15_DIR / "14_additive_primary_interaction_results.csv")
cell_counts = read_csv(MODEL15_DIR / "09_additive_vitd_dosage_cell_counts.csv")
ref_alt = read_csv(STEP17_DIR / "17_tier1_vdr_aou_ref_alt_records_for_harmonization.csv")
pooled_qc = read_csv(STEP17_DIR / "17_tier1_vdr_pooled_genotype_qc_hwe_descriptive.csv")
ld_pairs = read_csv(STEP17_DIR / "17_tier1_vdr_pairwise_ld_r2_dprime_unphased_em.csv")

for df in [cvd_primary, pooled_qc, ld_pairs]:
    for col in df.columns:
        if col not in {"rsid", "snp1", "snp2", "status", "model_dataset", "term", "analysis_family", "analysis_role", "model_name", "dosage", "interaction", "backend", "multiple_testing_family", "scope", "group", "run_id"}:
            df[col] = pd.to_numeric(df[col], errors="ignore")

print("Loaded inputs:")
print("  hail_validation", hail_validation.shape)
print("  cvd_primary", cvd_primary.shape)
print("  pooled_qc", pooled_qc.shape)
print("  ld_pairs", ld_pairs.shape)


## Genotype extraction completeness


In [ ]:
availability_rows = []
for rsid in SNP_ORDER:
    row = hail_validation.loc[hail_validation["check"] == f"{rsid}_nonmissing"]
    if not row.empty:
        availability_rows.append({"rsid": rsid, "nonmissing": int(row.iloc[0]["detail"])})
availability = pd.DataFrame(availability_rows)
wide_n = int(hail_validation.loc[hail_validation["check"] == "wide_row_count", "detail"].iloc[0])
complete_n = int(hail_validation.loc[hail_validation["check"] == "complete_four_snp_count", "detail"].iloc[0])
availability["missing"] = wide_n - availability["nonmissing"]
availability["label"] = availability["rsid"].map(SNP_LABELS)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
x = np.arange(len(availability))
ax.bar(x, availability["nonmissing"], color="#4C78A8", label="Non-missing genotype")
ax.bar(x, availability["missing"], bottom=availability["nonmissing"], color="#D9D9D9", label="Missing genotype")
ax.axhline(complete_n, color="#E45756", linestyle="--", linewidth=1.6, label=f"Complete all four SNPs: {complete_n:,}")
ax.set_xticks(x)
ax.set_xticklabels(availability["label"])
ax.set_ylabel("Participants")
ax.set_title("AoU VDR genotype extraction completeness")
ax.set_ylim(0, wide_n * 1.08)
for i, row in availability.iterrows():
    ax.text(i, row["nonmissing"] + wide_n * 0.012, f"{row['nonmissing']:,}", ha="center", va="bottom", fontsize=8)
ax.text(-0.45, wide_n * 1.02, f"Wide genotype table: {wide_n:,} participants", ha="left", va="center", fontsize=9)
ax.legend(frameon=False, loc="lower left")
fig.tight_layout()
save_figure(
    fig,
    "fig01_genotype_extraction_completeness",
    "Genotype extraction completeness for the four Tier 1 VDR SNPs. Additive ALT dosage was available for over 112,000 analytic participants at each SNP, with 112,252 participants complete for all four SNPs.",
    "supplement"
)
plt.show()


## CVD interaction forest


In [ ]:
forest = cvd_primary.copy()
forest["rsid"] = pd.Categorical(forest["rsid"], categories=SNP_ORDER, ordered=True)
forest = forest.sort_values("rsid")
forest["label"] = forest["rsid"].astype(str).map(SNP_LABELS)

y = np.arange(len(forest))[::-1]
fig, ax = plt.subplots(figsize=(7.4, 4.6))
hr = forest["hr"].astype(float).values
lo = forest["hr_ci_lower"].astype(float).values
hi = forest["hr_ci_upper"].astype(float).values
xerr = np.vstack([hr - lo, hi - hr])
colors = ["#4C78A8" if p >= 0.05 else "#F58518" for p in forest["p"].astype(float)]
ax.errorbar(hr, y, xerr=xerr, fmt="none", ecolor="#555555", elinewidth=1.4, capsize=3, zorder=1)
ax.scatter(hr, y, s=72, c=colors, edgecolor="white", linewidth=0.8, zorder=2)
ax.axvline(1.0, color="#222222", linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(forest["label"])
ax.set_xlabel("Hazard-ratio ratio for vitamin D deficiency per additional AoU ALT allele")
ax.set_title("Primary additive VDR-by-vitamin D deficiency CVD interaction models")
ax.set_xlim(0.78, 1.28)
for i, row in enumerate(forest.iloc[::-1].itertuples(index=False)):
    pass
for ypos, row in zip(y, forest.itertuples(index=False)):
    text = f"HR {row.hr:.3f} ({row.hr_ci_lower:.3f}-{row.hr_ci_upper:.3f}); p={row.p:.3f}; Bonf. p={row.p_bonferroni_4:.3f}"
    ax.text(1.285, ypos, text, ha="left", va="center", fontsize=8, clip_on=False)
ax.text(0.78, -0.72, "All models adjusted for age, female sex, winter, and BMI; no Bonferroni-significant interaction.", ha="left", va="center", fontsize=8)
fig.subplots_adjust(right=0.62)
save_figure(
    fig,
    "fig02_primary_additive_cvd_interaction_forest",
    "Primary additive Cox interaction results for vitamin D deficiency by VDR ALT dosage. Estimates are hazard-ratio ratios per additional selected AoU ALT allele; none were significant after correction for four Tier 1 interaction tests.",
    "main"
)
plt.show()


## Pairwise LD


In [ ]:
def pair_matrix(value_col):
    mat = pd.DataFrame(np.eye(len(SNP_ORDER)), index=SNP_ORDER, columns=SNP_ORDER, dtype=float)
    for row in ld_pairs.itertuples(index=False):
        mat.loc[row.snp1, row.snp2] = float(getattr(row, value_col))
        mat.loc[row.snp2, row.snp1] = float(getattr(row, value_col))
    return mat

r2_mat = pair_matrix("r2_corr")
dp_mat = pair_matrix("d_prime_em")
labels = [SNP_SHORT[x] + "\n" + x for x in SNP_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
for ax, mat, title, vmax in [
    (axes[0], r2_mat, "Allele-count r2", 1.0),
    (axes[1], dp_mat, "Unphased EM D-prime", 1.0),
]:
    im = ax.imshow(mat.values, vmin=0, vmax=vmax, cmap="Blues")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = mat.values[i, j]
            text_color = "white" if val > 0.55 else "#222222"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=text_color)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("AoU VDR pairwise LD among four Tier 1 SNPs", y=1.03)
fig.tight_layout()
save_figure(
    fig,
    "fig05_vdr_pairwise_ld_heatmaps",
    "Pairwise LD among the four Tier 1 VDR SNPs. BsmI/ApaI/TaqI are correlated 3-prime markers, whereas FokI/rs2228570 is nearly independent of that cluster. D-prime values are unphased EM estimates and do not imply phased haplotype testing.",
    "main_or_supplement"
)
plt.show()


## Pooled genotype QC


In [ ]:
qc = pooled_qc.copy()
qc["rsid"] = pd.Categorical(qc["rsid"], categories=SNP_ORDER, ordered=True)
qc = qc.sort_values("rsid")
qc["label"] = qc["rsid"].astype(str).map(SNP_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.2))
x = np.arange(len(qc))
axes[0].bar(x, qc["maf"].astype(float), color="#4C78A8")
axes[0].set_xticks(x)
axes[0].set_xticklabels(qc["label"])
axes[0].set_ylabel("Minor allele frequency")
axes[0].set_title("Pooled MAF")
axes[0].set_ylim(0, 0.55)
for i, val in enumerate(qc["maf"].astype(float)):
    axes[0].text(i, val + 0.015, f"{val:.3f}", ha="center", va="bottom", fontsize=8)

axes[1].bar(x, qc["missing_pct"].astype(float), color="#F58518")
axes[1].set_xticks(x)
axes[1].set_xticklabels(qc["label"])
axes[1].set_ylabel("Missing genotype (%)")
axes[1].set_title("Pooled missingness")
axes[1].set_ylim(0, max(qc["missing_pct"].astype(float)) * 1.8)
for i, val in enumerate(qc["missing_pct"].astype(float)):
    axes[1].text(i, val + 0.001, f"{val:.3f}%", ha="center", va="bottom", fontsize=8)

fig.suptitle("Descriptive genotype QC for selected AoU VDR records", y=1.03)
fig.text(0.5, -0.02, "Pooled HWE p-values are descriptive only; ancestry-unrelated/sample-QC HWE was not computed in the current analysis.", ha="center", va="top", fontsize=8)
fig.tight_layout()
save_figure(
    fig,
    "fig06_pooled_genotype_qc_maf_missingness",
    "Descriptive pooled genotype QC for the selected AoU VDR records. Missingness was low and MAF was common for all four SNPs; pooled HWE was not interpreted as pass/fail because genetic ancestry, unrelated-set, and sample-QC filters were not applied.",
    "supplement"
)
plt.show()


## Write figure captions


In [ ]:
manifest = pd.DataFrame(FIGURE_MANIFEST)
manifest_path = OUTPUT_DIR / "18_q3_vdr_manuscript_figure_manifest.csv"
manifest.to_csv(manifest_path, index=False)

caption_path = OUTPUT_DIR / "18_q3_vdr_manuscript_figure_captions.md"
with open(caption_path, "w", encoding="utf-8") as f:
    f.write("# Q3 VDR Manuscript Figure Captions\n\n")
    f.write(f"Generated: {datetime.now().isoformat(timespec='seconds')}\n\n")
    for i, row in manifest.iterrows():
        f.write(f"## {row['figure_stem']}\n\n")
        f.write(row["caption"] + "\n\n")
        f.write(f"PNG: `{row['png']}`\n\n")
        f.write(f"PDF: `{row['pdf']}`\n\n")

print("Wrote manifest:", manifest_path)
print("Wrote captions:", caption_path)
manifest
